# CoRAL-Sep Stage 1: Single Adapter Training

Train one LoRA adapter (`reverb`, `noise`, or `codec`) on a T4/P100 GPU.
Expected time per adapter: **6–8 hours**.

Run three times (once per adapter). Stage 1 LR = **1e-4**.

## Setup checklist
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Set `ADAPTER_NAME` below to `reverb`, `noise`, or `codec`.
3. Mount Google Drive or set `DATA_ROOT` to your data path.
4. Set `CHECKPOINT_DIR` where adapters will be saved.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
ADAPTER_NAME = "reverb"        # reverb | noise | codec
DATA_ROOT    = "/kaggle/input/calmsep-8k"   # adjust to your dataset path
CHECKPOINT_DIR = "/kaggle/working/adapters"
CONFIG_PATH  = "configs/adapters/reverb.yaml"  # or noise.yaml / codec.yaml
EPOCHS       = 40
BATCH_SIZE   = 8
LR           = 1e-4
DEVICE       = "cuda"          # or cpu for smoke-testing
NOISE_DIR      = "/kaggle/input/calmsep-8k/noise"
RIR_BANK       = "/kaggle/input/calmsep-8k/rirs/bank.json"
BUT_DIR        = "/kaggle/input/calmsep-8k/but-reverbdb"


In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q soundfile scipy huggingface_hub transformers

In [ ]:
# ── Clone / mount repo ────────────────────────────────────────────────────────
import os
import sys

# If running in Kaggle, the repo must be uploaded as a dataset or cloned.
# Adjust this path to where the CoRAL-Sep codebase is located.
REPO_PATH = "/kaggle/input/calmsep-code"  # update as needed
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────────────────────
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Run Stage 1 training ──────────────────────────────────────────────────────
import subprocess
import sys

cmd = [
    sys.executable, "-m", "train.stage1_single",
    "--adapter", ADAPTER_NAME,
    "--data-root", DATA_ROOT,
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--config", CONFIG_PATH,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--device", DEVICE,
    "--noise-dir", NOISE_DIR,
    "--rir-bank",  RIR_BANK,
    "--bf16",
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_PATH, capture_output=False)
print(f"\nExit code: {result.returncode}")

In [ ]:
# ── Inspect saved adapter ─────────────────────────────────────────────────────
adapter_path = os.path.join(CHECKPOINT_DIR, f"{ADAPTER_NAME}_adapter.pt")
if os.path.exists(adapter_path):
    state = torch.load(adapter_path, map_location="cpu")
    print(f"Adapter keys: {list(state.keys())[:5]}...")
    n_params = sum(v.numel() for v in state.values())
    print(f"Total adapter params: {n_params:,}")
else:
    print(f"Adapter not found at {adapter_path}")

## Next steps

1. Run this notebook 3 times with `ADAPTER_NAME` = `reverb`, `noise`, `codec`.
2. Collect `{adapter}_adapter.pt` files in `CHECKPOINT_DIR`.
3. Run `stage2_universal.ipynb` to train the universal adapter baseline.
4. Run `stage3_gate.ipynb` after Stage 1 + 2 are done.